In [4]:
import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [5]:
import sys
import importlib

sys.path.append("../src")

import revenue_engine
importlib.reload(revenue_engine)

from revenue_engine import *

In [7]:
records = []

states = [
    "Punjab",
    "Haryana",
    "Uttar Pradesh",
    "Madhya Pradesh",
    "Bihar"
]

crops = [
    "Wheat",
    "Rice"
]

for i in range(2000):

    crop = random.choice(crops)

    state = random.choice(states)

    land_area = round(random.uniform(1,10),2)

    yield_per_acre = round(random.uniform(1.5,4),2)

    predicted_price = round(random.uniform(18000,30000),2)

    cultivation_cost = round(random.uniform(50000,250000),2)

    loan_amount = round(random.uniform(30000,300000),2)

    interest_rate = round(random.uniform(7,12),2)

    tenure = random.choice([6,9,12])

    production = compute_production(
        land_area,
        yield_per_acre
    )

    revenue = compute_revenue(
        production,
        predicted_price
    )

    profit = compute_profit(
        revenue,
        cultivation_cost
    )

    emi = compute_emi(
        loan_amount,
        interest_rate,
        tenure
    )

    margin = compute_profit_margin(
        profit,
        revenue
    )

    ratio = compute_debt_to_profit_ratio(
        loan_amount,
        profit
    )

    if profit > 120000 and ratio < 2:
        risk = "SAFE"

    elif profit > 50000 and ratio < 4:
        risk = "MODERATE"

    else:
        risk = "RISKY"

    records.append([
        crop,
        state,
        land_area,
        yield_per_acre,
        predicted_price,
        cultivation_cost,
        production,
        revenue,
        profit,
        loan_amount,
        interest_rate,
        tenure,
        emi,
        margin,
        ratio,
        risk
    ])

columns = [

    "Crop",

    "State",

    "Land_Area",

    "Yield_per_Acre",

    "Predicted_Price",

    "Cultivation_Cost",

    "Production",

    "Revenue",

    "Profit",

    "Loan_Amount",

    "Interest_Rate",

    "Tenure",

    "EMI",

    "Profit_Margin",

    "Debt_to_Profit_Ratio",

    "Risk"

]

loan_df = pd.DataFrame(records,columns=columns)

loan_df.head()

,Crop,State,Land_Area,Yield_per_Acre,Predicted_Price,Cultivation_Cost,Production,Revenue,Profit,Loan_Amount,Interest_Rate,Tenure,EMI,Profit_Margin,Debt_to_Profit_Ratio,Risk
0,Rice,Bihar,4.04,2.53,28903.96,58515.64,10.22,295398.47,236882.83,273962.88,7.75,9,31431.73,80.19,1.16,SAFE
1,Wheat,Madhya Pradesh,8.79,3.61,23825.37,186636.65,31.73,755978.99,569342.34,248062.59,9.33,6,42476.09,75.31,0.44,SAFE
2,Wheat,Punjab,9.43,2.91,20536.86,176790.93,27.44,563531.44,386740.51,209958.10,8.26,9,24138.92,68.63,0.54,SAFE
3,Wheat,Uttar Pradesh,6.77,1.76,27833.63,205571.55,11.92,331776.87,126205.32,250722.75,7.19,6,42667.80,38.04,1.99,SAFE
4,Wheat,Bihar,3.79,2.62,23212.33,176090.96,9.93,230498.44,54407.48,127213.32,7.46,12,11034.35,23.60,2.34,MODERATE


In [8]:
loan_df.to_csv(
    "../data/synthetic_loan_data.csv",
    index=False
)

print("Synthetic Dataset Saved Successfully")

Synthetic Dataset Saved Successfully


In [9]:
loan_df = pd.read_csv("../data/synthetic_loan_data.csv")

print("Dataset Loaded Successfully")

loan_df.head()

Dataset Loaded Successfully


,Crop,State,Land_Area,Yield_per_Acre,Predicted_Price,Cultivation_Cost,Production,Revenue,Profit,Loan_Amount,Interest_Rate,Tenure,EMI,Profit_Margin,Debt_to_Profit_Ratio,Risk
0,Rice,Bihar,4.04,2.53,28903.96,58515.64,10.22,295398.47,236882.83,273962.88,7.75,9,31431.73,80.19,1.16,SAFE
1,Wheat,Madhya Pradesh,8.79,3.61,23825.37,186636.65,31.73,755978.99,569342.34,248062.59,9.33,6,42476.09,75.31,0.44,SAFE
2,Wheat,Punjab,9.43,2.91,20536.86,176790.93,27.44,563531.44,386740.51,209958.10,8.26,9,24138.92,68.63,0.54,SAFE
3,Wheat,Uttar Pradesh,6.77,1.76,27833.63,205571.55,11.92,331776.87,126205.32,250722.75,7.19,6,42667.80,38.04,1.99,SAFE
4,Wheat,Bihar,3.79,2.62,23212.33,176090.96,9.93,230498.44,54407.48,127213.32,7.46,12,11034.35,23.60,2.34,MODERATE


In [10]:
# Target column
target = "Risk"

# Features
X = loan_df.drop(columns=[target])

# Labels
y = loan_df[target]

# Numeric columns
numeric_features = X.select_dtypes(include=np.number).columns

# Categorical columns
categorical_features = X.select_dtypes(include="object").columns

print("Numeric Features")
print(numeric_features)

print("\nCategorical Features")
print(categorical_features)

Numeric Features
Index(['Land_Area', 'Yield_per_Acre', 'Predicted_Price', 'Cultivation_Cost',
       'Production', 'Revenue', 'Profit', 'Loan_Amount', 'Interest_Rate',
       'Tenure', 'EMI', 'Profit_Margin', 'Debt_to_Profit_Ratio'],
      dtype='object')

Categorical Features
Index(['Crop', 'State'], dtype='object')


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

Training Shape : (1600, 15)
Testing Shape : (400, 15)


In [12]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

risk_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

risk_model.fit(X_train, y_train)

print("Risk Model Trained Successfully")

Risk Model Trained Successfully


In [26]:
import numpy as np

print(np.isinf(loan_df.select_dtypes(include=np.number)).sum())

Land_Area                 0
Yield_per_Acre            0
Predicted_Price           0
Cultivation_Cost          0
Production                0
Revenue                   0
Profit                    0
Loan_Amount               0
Interest_Rate             0
Tenure                    0
EMI                       0
Profit_Margin             0
Debt_to_Profit_Ratio    353
dtype: int64


In [27]:
import numpy as np

print("Any infinite values in X?")
print(np.isinf(X.select_dtypes(include=np.number)).sum())

print("\nMaximum values:")
print(X.select_dtypes(include=np.number).max())

Any infinite values in X?
Land_Area                 0
Yield_per_Acre            0
Predicted_Price           0
Cultivation_Cost          0
Production                0
Revenue                   0
Profit                    0
Loan_Amount               0
Interest_Rate             0
Tenure                    0
EMI                       0
Profit_Margin             0
Debt_to_Profit_Ratio    353
dtype: int64

Maximum values:
Land_Area                    10.00
Yield_per_Acre                4.00
Predicted_Price           29999.62
Cultivation_Cost         249691.06
Production                   38.60
Revenue                 1016038.46
Profit                   938249.83
Loan_Amount              299951.97
Interest_Rate                12.00
Tenure                       12.00
EMI                       51639.89
Profit_Margin                94.59
Debt_to_Profit_Ratio           inf
dtype: float64


In [23]:
print("Any infinite values in X_train?")
print(np.isinf(X_train.select_dtypes(include=np.number)).sum())

Any infinite values in X_train?
Land_Area               0
Yield_per_Acre          0
Predicted_Price         0
Cultivation_Cost        0
Production              0
Revenue                 0
Profit                  0
Loan_Amount             0
Interest_Rate           0
Tenure                  0
EMI                     0
Profit_Margin           0
Debt_to_Profit_Ratio    0
dtype: int64


In [14]:
print("Missing values in X:")
print(X.isnull().sum())

print("\nMissing values in X_train:")
print(X_train.isnull().sum())

Missing values in X:
Crop                    0
State                   0
Land_Area               0
Yield_per_Acre          0
Predicted_Price         0
Cultivation_Cost        0
Production              0
Revenue                 0
Profit                  0
Loan_Amount             0
Interest_Rate           0
Tenure                  0
EMI                     0
Profit_Margin           0
Debt_to_Profit_Ratio    0
dtype: int64

Missing values in X_train:
Crop                    0
State                   0
Land_Area               0
Yield_per_Acre          0
Predicted_Price         0
Cultivation_Cost        0
Production              0
Revenue                 0
Profit                  0
Loan_Amount             0
Interest_Rate           0
Tenure                  0
EMI                     0
Profit_Margin           0
Debt_to_Profit_Ratio    0
dtype: int64


In [30]:
print(X.shape)
print(X_train.shape)

print(np.isinf(X.select_dtypes(include=np.number)).sum())

print(np.isinf(X_train.select_dtypes(include=np.number)).sum())

(2000, 15)
(1600, 15)
Land_Area                 0
Yield_per_Acre            0
Predicted_Price           0
Cultivation_Cost          0
Production                0
Revenue                   0
Profit                    0
Loan_Amount               0
Interest_Rate             0
Tenure                    0
EMI                       0
Profit_Margin             0
Debt_to_Profit_Ratio    353
dtype: int64
Land_Area                 0
Yield_per_Acre            0
Predicted_Price           0
Cultivation_Cost          0
Production                0
Revenue                   0
Profit                    0
Loan_Amount               0
Interest_Rate             0
Tenure                    0
EMI                       0
Profit_Margin             0
Debt_to_Profit_Ratio    286
dtype: int64


In [15]:
import numpy as np
print(np.isinf(loan_df["Debt_to_Profit_Ratio"]).sum())

0


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (1600, 15)
Testing Shape: (400, 15)


In [22]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

risk_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

risk_model.fit(X_train, y_train)

print("Risk Model Trained Successfully")

Risk Model Trained Successfully


In [18]:
predictions = risk_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

Accuracy: 0.9950

Classification Report
              precision    recall  f1-score   support

    MODERATE       0.98      0.98      0.98        48
       RISKY       0.99      1.00      1.00       106
        SAFE       1.00      1.00      1.00       246

    accuracy                           0.99       400
   macro avg       0.99      0.99      0.99       400
weighted avg       1.00      0.99      1.00       400


Confusion Matrix
[[ 47   1   0]
 [  0 106   0]
 [  1   0 245]]


In [21]:
joblib.dump(risk_model, "../models/risk_model.pkl")

print(" Risk Model Saved Successfully!")

 Risk Model Saved Successfully!


In [20]:
sample = X.iloc[[0]]

prediction = risk_model.predict(sample)

print("Predicted Risk:", prediction[0])

Predicted Risk: SAFE


In [1]:
import os

print(os.getcwd())

C:\Users\Acer\Documents\price predictor\notebooks
